# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's inspect the available record sets and their fields by their `@id` references.

In [ ]:
# List all available record sets by `@id`
print("Record Sets (referenced by @id):")
for record_set in dataset.metadata.record_sets:
    print(f"  RecordSet @id: {record_set['@id']}, name: {record_set.get('name', '')}")
    print("    Fields:")
    for field in record_set.get('field', []):
        # Each field is either a dict or a @id string
        if isinstance(field, dict):
            print(f"      Field @id: {field.get('@id', '')}, name: {field.get('name', '')}")
        else:
            print(f"      Field @id: {field}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this notebook, we load all record sets by their @id.

# First, collect all record set @id values
record_sets_ids = [rs['@id'] for rs in dataset.metadata.record_sets]

dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set: {record_set_id} (shape: {df.shape})")

# Show columns for first non-empty record set
for rec_id, df in dataframes.items():
    print(f"Columns for {rec_id}:", df.columns.tolist())
    display(df.head())
    break  # Only show one as an example

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

We'll select a numeric field from the first record set for demonstration.

In [ ]:
# Pick the first record set with a numeric field for EDA
selected_record_set = None
numeric_field = None

for rs in dataset.metadata.record_sets:
    rec_id = rs['@id']
    df = dataframes.get(rec_id)
    if df is not None:
        # Attempt to find a numeric field by declared dataType in schema
        numeric_candidates = []
        for field in rs.get('field', []):
            if isinstance(field, dict):
                dtype = field.get('dataType', '').lower()
                if 'int' in dtype or 'float' in dtype or 'number' in dtype:
                    numeric_candidates.append(field['@id'])
            # Some fields might be referenced by @id only
            # To fully resolve, we'd need to dereference which field object
        # Fallback: Guess numeric fields by DataFrame dtype
        if not numeric_candidates:
            for col in df.columns:
                try:
                    if pd.api.types.is_numeric_dtype(df[col]):
                        numeric_candidates.append(col)
                except Exception:
                    continue
        if numeric_candidates:
            selected_record_set = rec_id
            numeric_field = numeric_candidates[0]
            break

if selected_record_set is not None and numeric_field is not None:
    print(f"Selected record set: {selected_record_set}")
    print(f"Selected numeric field: {numeric_field}")

    # Ensure dtype for numeric analysis
    df = dataframes[selected_record_set].copy()
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

    # Filtering: Remove NaNs, filter for values above threshold
    threshold = df[numeric_field].quantile(0.9)  # Use 90th percentile as example
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (showing up to 5 rows):")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a non-numeric field
    group_field = None
    for col in df.columns:
        if col != numeric_field and not pd.api.types.is_numeric_dtype(df[col]):
            group_field = col
            break

    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean {numeric_field} by {group_field}:")
        display(grouped_df.head())
else:
    print("No suitable record set with a numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set is not None and numeric_field is not None:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    df = dataframes[selected_record_set]
    # Histogram of numeric field
    df[numeric_field].hist(ax=ax[0], bins=20)
    ax[0].set_title(f"Distribution of {numeric_field}")
    ax[0].set_xlabel(numeric_field)
    ax[0].set_ylabel('Count')

    # Boxplot grouped by a (first non-numeric) group field
    if group_field is not None:
        sns.boxplot(x=group_field, y=numeric_field, data=df, ax=ax[1])
        ax[1].set_title(f"{numeric_field} by {group_field}")
        ax[1].tick_params(axis='x', rotation=45)
    else:
        ax[1].set_visible(False)
    plt.show()
else:
    print("No suitable numeric column for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.
- We loaded the Mass Spectrometry dataset using its Croissant schema and explored the available record sets and fields via their `@id`s.
- We demonstrated typical data extraction, filtering, normalization, and grouping using pandas.
- The visualizations show the distribution and group structure within a selected numeric field.

Further analysis can be performed using the field `@id`s for reproducible and schema-driven workflows.